In [ ]:
import pandas as pd
import datetime as dt

In [48]:
paths = ({
    'invoice_level_table': '../data/D3_invoice-level table.csv',
    'top_3_invoices': '../outputs/D4_top_3_invoices.csv',
})

In [ ]:
df = pd.read_parquet(paths['cleaned_dataset'])
df_invoice = pd.read_csv(paths['invoice_level_table'])

#### 1. Revenue aggregationn

In [27]:
df_agg = df_invoice.filter(['country', 'invoice_first_date', 'invoice_revenue'])
df_agg['invoice_first_date'] = pd.to_datetime(df_agg['invoice_first_date'])
df_agg['Month'] = df_agg['invoice_first_date'].dt.to_period('M')

# by country
df_country = df_agg.groupby(['country'])['invoice_revenue'].sum()
# by month
df_month = df_agg.groupby(['Month'])['invoice_revenue'].sum()

In [30]:
# by (country, month)
# method1: pivot table format (multi-indexing)
df_agg.groupby(['country', 'Month']).agg(
    revenue=('invoice_revenue', 'sum'),
)

# method2: reset index: index shown as multiple columns
df_agg.groupby(['country', 'Month']).agg(
    revenue=('invoice_revenue', 'sum'),
).reset_index()

,country,Month,revenue
0,Australia,2010-12,1032.85
1,Australia,2011-01,9017.71
2,Australia,2011-02,14695.42
3,Australia,2011-03,17223.99
4,Australia,2011-04,771.60
...,...,...,...
282,Unspecified,2011-04,299.10
283,Unspecified,2011-05,852.68
284,Unspecified,2011-06,185.78
285,Unspecified,2011-07,798.48


#### 2. sorting using `.sort_values()`

In [31]:
df_agg.sort_values("invoice_revenue", ascending=False)

,country,invoice_first_date,invoice_revenue,Month
18499,United Kingdom,2011-12-09 09:15:00,168469.60,2011-12
1909,United Kingdom,2011-01-18 10:01:00,77183.60,2011-01
7925,United Kingdom,2011-06-10 15:28:00,38970.00,2011-06
12417,United Kingdom,2011-09-20 11:05:00,31698.16,2011-09
8111,Australia,2011-06-15 13:37:00,22775.93,2011-06
...,...,...,...,...
794,United Kingdom,2010-12-10 10:52:00,0.95,2010-12
1728,United Kingdom,2011-01-12 12:41:00,0.85,2011-01
2379,United Kingdom,2011-01-31 15:37:00,0.55,2011-01
12592,United Kingdom,2011-09-22 14:18:00,0.40,2011-09


#### 4. calculate per country revenue stats

In [36]:
df_country = df_agg.groupby(['country']).agg(
    country_avg_aov=('invoice_revenue', 'mean'),
    country_median_aov=('invoice_revenue', 'median')
)

#### 5. Top 3 invoices per country

In [38]:
df_summary = pd.merge(df_invoice, df_country, on='country', how='left')
df_summary['aov_vs_country_avg'] = df_summary['invoice_revenue'] / df_summary['country_avg_aov']

In [49]:
top_3_invoices = df_summary.groupby('country').apply(lambda x: x.nlargest(3, 'aov_vs_country_avg')).reset_index()
top_3_invoices.to_csv(paths['top_3_invoices'], index=False)